# S3 Demo: LLMs via Amazon Bedrock (Topic 3.2)
Same prompt, different models & parameters. **Model IDs live in `config/course_config.py`** : 
code against config, not literals (IDs churn quarterly).
No Bedrock in your sandbox? `USE_MOCK_AWS=1` replays recorded responses.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [ ]:
from course_utils import aws
import importlib, course_config as cfg   # config dir is on sys.path via course_utils.aws
print('fast :', cfg.BEDROCK_MODEL_FAST)
print('strong:', cfg.BEDROCK_MODEL_STRONG, '| mock:', cfg.USE_MOCK_AWS)

In [ ]:
review = """I have been a customer for three years and loved the service until this spring.
Two price increases, two support tickets closed without resolution, and now my
internet drops daily. The tech quality used to be excellent. I am now comparing
competitor offers and will switch within a month unless someone actually helps."""

prompt = "Summarize the following customer review in two sentences: " + review
print(aws.bedrock_chat(prompt))

## Same prompt, stronger model: quality vs. cost

In [ ]:
print(aws.bedrock_chat(prompt, model=cfg.BEDROCK_MODEL_STRONG))

## Temperature: determinism vs. variety

In [ ]:
title_prompt = "Give this customer review a 4-word title: " + review
for t in (0.0, 1.0):
    print(f'--- temperature {t} ---')
    for _ in range(2):
        print(aws.bedrock_chat(title_prompt, temperature=t))

## max_tokens: a hard stop, not a request to be brief

In [ ]:
reply_prompt = "Write a short, polite reply e-mail to this customer: " + review
text, usage = aws.bedrock_chat(reply_prompt, max_tokens=50, return_usage=True)
print(text)
print('\n', usage)   # stopReason 'max_tokens' = the model was cut off mid-sentence

## From toy to application: structured extraction (→ your task)

In [ ]:
schema_prompt = ("Extract the following fields as JSON (product, issue_category, "
                 "sentiment, urgency) from this text. Reply with ONLY the JSON.\n" + review)
print(aws.bedrock_chat(schema_prompt, temperature=0.0))
# Look at issue_category: we never gave a list of allowed values, so the model invented one.

## What does one call cost? Fast vs. strong

In [ ]:
rows = []
for tier, model in (("fast", cfg.BEDROCK_MODEL_FAST), ("strong", cfg.BEDROCK_MODEL_STRONG)):
    _, u = aws.bedrock_chat(prompt, model=model, return_usage=True)
    p_in, p_out = cfg.BEDROCK_PRICE_PER_1M[model]
    usd = (u["inputTokens"] * p_in + u["outputTokens"] * p_out) / 1_000_000
    rows.append((tier, model, u["inputTokens"], u["outputTokens"], usd))
    print(f"{tier:6} {model:24} in={u['inputTokens']:>4} out={u['outputTokens']:>4}  "
          f"${usd:.6f} per call  ->  ${usd * 1_000_000:,.0f} per million calls")
print(f"\nstrong / fast cost ratio: {rows[1][4] / rows[0][4]:.0f}x")

**Takeaways:** parameters are product decisions; explicit structure beats hope; temperature 0 for extraction; pick the cheapest model that is good enough.